Extracción de características (Embeddings)

In [ ]:
import os
import cv2
import numpy as np
import joblib
from deepface import DeepFace
from time import time

# Cargamos solo las primeras 600 imagenes de cada emoción
MAX_IMAGES_PER_CLASS = 600 

print(f"Extracción de Embeddings (MODO RÁPIDO: max {MAX_IMAGES_PER_CLASS} por clase)")

# --- Función LoadDataset ---
# Recorre el directorio de imágenes, procesa cada cara con DeepFace para obtener su "huella digital" numérica (embedding)
# y devuelve los arrays X (características) e Y (etiquetas) necesarios para entrenar el modelo SVM.
def LoadDataset(folder, ext, max_per_class):
    nclasses = 0        # Contador de clases (emociones)
    nperclass = []      # Lista para guardar cuántas imágenes hay por clase
    classlabels = []    # Lista para los nombres de las clases
    X = []      # Lista para guardar los 'embeddings' (los datos)
    Y = []      # La lista de etiquetas (ej: 0, 1, 2) que van con X

    print(f"Cargando dataset desde: {folder}")
    
    # Lista todas las subcarpetas
    class_list = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
    
    # Recorre las subcarpetas de emociones (ej: 'happy', 'sad'...)
    for class_name in class_list:
        class_folder = os.path.join(folder, class_name)
            
        nclasses += 1   # Sumamos una clase al contador
        nsamples = 0    # Reseteamos el contador de imágenes para esta nueva clase
        print(f"\nCargando clase: {class_name} ({nclasses}/{len(class_list)})")

        # Recorre las imágenes dentro de cada carpeta de emoción
        for file_name in os.listdir(class_folder):
            
            # Comprueba si ya hemos alcanzado el límite de imágenes
            if nsamples >= max_per_class:
                print(f"   ... Límite alcanzado ({max_per_class} imágenes)")
                break # Rompe el bucle de esta clase y pasa a la siguiente

            # Comprueba si el archivo es una imagen con la extensión correcta
            if file_name.endswith(ext):
                image_path = os.path.join(class_folder, file_name) # Ruta completa a la imagen
                try:
                    # 1. Leer imagen con OpenCV
                    image = cv2.imread(image_path)
                    if image is None:
                        continue
                    
                    # 2. Redimensionar: Facenet espera 160x160.
                    img1 = cv2.resize(image, dim, interpolation=cv2.INTER_AREA)

                    # 3. Usar la red neuronal 'Facenet' para "mirar" la imagen
                    # y convertirla en un vector de 128 números (el 'embedding').
                    embedding_objs = DeepFace.represent(
                        img_path=img1,
                        model_name=model_name,
                        enforce_detection=False # Asume que la imagen ya es una cara
                    )

                    # Extraemos solo el vector numérico
                    img_embedding = embedding_objs[0]["embedding"]
                    
                    # 4. Guardar datos en las listas
                    X.append(img_embedding) # Añade el vector lista de datos 'X'(features)
                    Y.append(nclasses - 1)  # Añade la etiqueta numérica (ej: 0 para 'angry') a nuestra lista 'Y'
                    nsamples += 1   # Suma 1 al contador de imágenes de esta clase
                    
                    # Imprime el progreso cada 100 imágenes (Feedback visual)
                    if nsamples % 100 == 0:
                        print(f"\r   ... procesadas {nsamples} imágenes", end="")
                
                except Exception as e:
                    # Ignora errores de 'represent' (cara no encontrada)
                    pass

        print(f"\r   -> Clase '{class_name}' completada. Total: {nsamples} imágenes.")
        nperclass.append(nsamples)      # Guarda el total de esta clase
        classlabels.append(class_name)  # Guarda el nombre de esta clase

    # Convertimos las listas de Python en 'numpy arrays' (necesario para Scikit-Learn)
    X = np.array(X, dtype='float32')
    Y = np.array(Y, dtype='float64')

    if X.size == 0:
        return X, Y, 0, 0, 0, [], [], []

    n_samples, n_features = X.shape     # n_samples = ~4200 (600*7), n_features = 128
    class_names = np.array(classlabels)
    n_classes = class_names.shape[0]    # n_classes = 7

    return X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names

# --- CONFIGURACIÓN DEL MODELO BASE ---
model_name = "Facenet" # Usamos Facenet
print(f"Construyendo modelo: {model_name}")
# Cargamos el modelo solo para obtener sus dimensiones de entrada requeridas (input_shape)
model = DeepFace.build_model(model_name)
dim = (model.input_shape[1], model.input_shape[0]) 
print(f"Dimensiones de entrada: {dim}")

# --- CARGA DEL DATASET ---
folder = "C:/Users/lucia/Downloads/train" 

X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names = LoadDataset(folder, '.png', MAX_IMAGES_PER_CLASS)

print("\n--- Información del Dataset ---")
print(f"# Muestras: {n_samples}")
print(f"# Características (Embeddings): {n_features}")
print(f"# Clases: {n_classes}")

# --- GUARDAR LOS DATOS EXTRAÍDOS ---
if n_samples > 0:
    print("\nGuardando datos extraídos en archivos .pkl...")
    
    joblib.dump(X, 'embeddings_X.pkl')      # Los datos matemáticos de las caras
    joblib.dump(Y, 'labels_Y.pkl')          # Qué emoción es cada cara
    joblib.dump(class_names, 'emotion_class_names.pkl')     # Los nombres ("feliz", "triste")
    
    print("¡Éxito! Archivos 'embeddings_X.pkl', 'labels_Y.pkl' y 'emotion_class_names.pkl' guardados.")
else:
    print("Error: No se cargaron muestras. Verifica la ruta de tu dataset ('folder') y la extensión ('.png').")

Script 1: Extracción de Embeddings (MODO RÁPIDO: max 600 por clase)
Construyendo modelo: Facenet
Dimensiones de entrada: (160, 160)
Cargando dataset desde: C:/Users/lucia/Downloads/train

Cargando clase: angry (1/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'angry' completada. Total: 600 imágenes.

Cargando clase: disgusted (2/7)
   -> Clase 'disgusted' completada. Total: 436 imágenes.

Cargando clase: fearful (3/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'fearful' completada. Total: 600 imágenes.

Cargando clase: happy (4/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'happy' completada. Total: 600 imágenes.

Cargando clase: neutral (5/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'neutral' completada. Total: 600 imágenes.

Cargando clase: sad (6/7)
   ... procesadas 600 imágenes   ... Límite alcanzado (600 imágenes)
   -> Clase 'sa

Entrenamiento

In [ ]:
import numpy as np
import joblib
from time import time
# MinMaxScaler: Para poner todos los datos en la misma escala (de 0 a 1)
from sklearn.preprocessing import MinMaxScaler
# SVC: Support Vector Classification
from sklearn.svm import SVC
# GridSearchCV: Herramienta para probar muchas configuraciones automáticamente y encontrar la mejor
from sklearn.model_selection import GridSearchCV

print("Entrenamiento del Modelo SVM (Rápido)")

# --- CARGA DE DATOS PREEXTRAÍDOS ---
try:
    print("Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...")
    # Arrays con las características y etiquetas del bloque anterior
    X = joblib.load('embeddings_X.pkl')
    Y = joblib.load('labels_Y.pkl')
    print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características.")
except FileNotFoundError:
    print("Error: No se encontraron los archivos .pkl.")
    exit()

# --- ENTRENAMIENTO DE MODELO  ---
if X.shape[0] > 0:
    # 1. Normalizar datos para un mejor funcionamiento
    print("\nEntrenando Scaler (MinMaxScaler)...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    print("Iniciando GridSearchCV para SVM...")
    t0 = time()

    # Definimos los hiperparámetros a probar. GridSearchCV probará todas las combinaciones.
    # 'C' (Regularización): Controla qué tan estricto es el modelo con los errores.
    # 'gamma' (Coeficiente del Kernel): Controla el radio de influencia de cada punto.
    parameters = {'C': [1e3, 5e3, 1e4], 
                  'gamma': [0.0001, 0.001, 0.01]} 
    
    # 2. Configuramos el entrenador:
    # kernel='rbf': Radial Basis Function. Permite clasificar datos que no son linealmente separables (curvas).
    # class_weight='balanced': Vital si tienes, por ejemplo, 600 fotos felices pero solo 200 tristes. Equilibra la importancia.
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced', probability=True), 
        parameters, 
        cv=3,       # Divide los datos en 3 para validar
        n_jobs=-1,  # Usar todos los cores
        verbose=3   # Imprime el progreso
    )
    # 3. El modelo busca la mejor forma matemática de separar las emociones.
    clf.fit(X_scaled, Y)
    
    print(f"GridSearchCV terminado en {time() - t0:.3f}s")
    print("Mejor estimador encontrado:")
    print(clf.best_estimator_) # Nos dice qué combinación de C y gamma ganó
    
    # -- GUARDADO DE MODELO --
    # Guardamos el "ganador" del GridSearchCV
    final_model = clf.best_estimator_
    print("\nGuardando modelos finales...")
    
    joblib.dump(final_model, 'emotion_svm_model.pkl')   # Guardamos el modelo entrenado
    joblib.dump(scaler, 'emotion_scaler.pkl')           # Guardamos el scaler
    
    print("¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.")
else:
    print("Error: Los datos cargados están vacíos.")

Script 2: Entrenamiento del Modelo SVM (Rápido)
Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...
Datos cargados: 4036 muestras, 128 características.

Entrenando Scaler (MinMaxScaler)...
Iniciando GridSearchCV para SVM...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
GridSearchCV terminado en 92.038s
Mejor estimador encontrado:
SVC(C=1000.0, class_weight='balanced', gamma=0.01, probability=True)

Guardando modelos finales...
¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.
¡Todo listo para ejecutar el prototipo!
